In [2]:
from steer_core.DataManager import DataManager

# Simplified imports using __init__.py
from steer_opencell_design import (
    PunchedCurrentCollector,
    AnodeFormulation,
    CathodeFormulation,
    Cathode, 
    Anode,
    Separator,
    ZFoldMonoLayer,
    ZFoldStack,
    SeparatorMaterial, 
    InsulationMaterial, 
    CurrentCollectorMaterial,
    TapeMaterial, 
    AnodeMaterial,
    CathodeMaterial,
    ConductiveAdditive,
    Binder,
    __version__
)

import pandas as pd
from datetime import datetime as dt

In [3]:
db = DataManager()

In [4]:
db.remove_data('cells', "name == 'Vendor D NFPP'")

In [5]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Cell 2,gASVQwYBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:38:51,0.4.9,Na/Na+
1,Cell 4,gASVBAABAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:39:18,0.4.9,Na/Na+
2,HiNa-NaCR32140-MP10,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:39:48,0.4.9,Na/Na+
3,QNAS-S40160NL,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:40:14,0.4.9,Na/Na+
4,QNAS-S40160RL,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:40:41,0.4.9,Na/Na+
5,UniGrid-NCO32140,gASVxQABAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:41:05,0.4.9,Na/Na+
6,Vendor E NFPP,gASVHwwBAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-12-01 14:41:54,0.4.9,Na/Na+
7,Vendor G NFM,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:42:20,0.4.9,Na/Na+
8,Vendor G NFPP,gASVbwsBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:42:42,0.4.9,Na/Na+
9,CBAK-32140NS,gASV7Q0AAAAAAACMOXN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-04 15:39:18,0.4.11,Na/Na+


In [6]:
db.get_table_names()

['cells',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'prismatic_container_materials',
 'separator_materials',
 'tape_materials',
 'binder_materials',
 'cathode_materials',
 'anode_materials']

In [7]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation_material = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polyethylene")

In [8]:
# Create the cathode

cathode_current_collector = PunchedCurrentCollector(
    material=current_collector_material,
    width=166,
    height=189,
    thickness=13,
    tab_width=50,
    tab_height=20,
    tab_position=50,
    coated_tab_height=4,
    insulation_width=5
)

cathode_active_material = CathodeMaterial.from_database("NFPP")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=1.88,
    mass_loading=18.12,
    insulation_material=insulation_material,
    insulation_thickness=3,
)



In [9]:
# Create the anode object

anode_current_collector = PunchedCurrentCollector(
    material=current_collector_material,
    width=168,
    height=191,
    thickness=13,
    tab_width=50,
    tab_height=20,
    tab_position=118,
    coated_tab_height=4,
    insulation_width=5
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=9,
    insulation_material=insulation_material,
    insulation_thickness=3
)



In [10]:
# make the layup

separator = Separator(
    material=separator_material,
    width=192,
    thickness=25
)

my_monolayer = ZFoldMonoLayer(
    anode=my_anode,
    cathode=my_cathode,
    separator=separator
)


In [11]:
# make the stack

my_stack = ZFoldStack(
    layup=my_monolayer,
    n_layers=40,
    name="Vendor D NFPP",
    additional_separator_wraps=6
)

In [12]:
print(f"Stack cost: {my_stack.cost:.2f} USD")
print(f"Stack mass: {my_stack.mass:.2f} g")

Stack cost: 9.77 USD
Stack mass: 874.22 g


In [17]:
my_stack.get_side_view(width=1600, height=500).show(renderer="chrome")

In [14]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_stack.name],
    'object': [my_stack.serialize()],
    'form_factor': ['Prismatic'],
    'internal_construction': ['Stacked'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [15]:
db.get_data(table_name='cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Cell 2,gASVQwYBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:38:51,0.4.9,Na/Na+
1,Cell 4,gASVBAABAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:39:18,0.4.9,Na/Na+
2,HiNa-NaCR32140-MP10,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:39:48,0.4.9,Na/Na+
3,QNAS-S40160NL,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:40:14,0.4.9,Na/Na+
4,QNAS-S40160RL,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:40:41,0.4.9,Na/Na+
5,UniGrid-NCO32140,gASVxQABAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:41:05,0.4.9,Na/Na+
6,Vendor E NFPP,gASVHwwBAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-12-01 14:41:54,0.4.9,Na/Na+
7,Vendor G NFM,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:42:20,0.4.9,Na/Na+
8,Vendor G NFPP,gASVbwsBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:42:42,0.4.9,Na/Na+
9,CBAK-32140NS,gASV7Q0AAAAAAACMOXN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-04 15:39:18,0.4.11,Na/Na+
